# Graph Theory Metrics from Functional Connectivity
## Network Analysis Tutorial

**Goal:** Convert a parcel-level FC matrix into a brain graph and compute node-level and graph-level network metrics.

```
FC matrix (CSV)  ──►  threshold  ──►  NetworkX graph  ──►  node metrics  ──►  threshold sweep  ──►  CSV
```

---
**How this notebook is structured**

| Section | What to do |
|---------|------------|
| 1. Imports | Run as-is |
| 2. Configuration | Fill in your FC matrix path and settings |
| 3. Explore the helpers | Run `fc_help()` to see available functions |
| 4 – 12. Analysis | Each cell has a `# YOUR CODE HERE` block to complete |
| 13. Answer key | Scroll down for a complete working solution |

---
**Prerequisites:** Complete the Functional Connectivity tutorial first.
This notebook starts from the FC matrix CSV saved by `save_fc_matrix()`.  
It reuses `fc_helpers.py` — make sure it is in your Python path.

*Helper functions: see `fc_helpers.py` — call `fc_help("function_name")` for detailed docs.*

## 1. Imports

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import networkx as nx
from nilearn import plotting

# Import all helper functions + the help browser
from fc_helpers import *

warnings.filterwarnings('ignore')
%matplotlib inline
print('Imports OK')

# nice for testing / debugging helper functions
import importlib, fc_helpers
importlib.reload(fc_helpers)
from fc_helpers import *

# Now confirm it's the right one
print(fc_helpers.__file__)

## 2. Configuration

Set every path and parameter here. Nothing downstream should need editing.

In [ ]:
# ── Subject / run identifiers ────────────────────────────────────────────────
SUBJECT  = 'sub-XXX'
SESSION  = 'ses-XXX'       # set to None if no session level
TASK     = 'rest'         # 'rest' for resting-state
SPACE    = 'MNI152NLin6Asym'

# ── Input: FC matrix from the FC tutorial ────────────────────────────────────
# Point this at the CSV file saved by save_fc_matrix() in the FC tutorial.
# The CSV has parcel labels as both row index and column headers.
FC_MATRIX_PATH = '/path/to/results/sub-XXX_ses-XXX_task-rest_atlas-schaefer200_fc-fisherz.csv'


# ── Parcellation (for visualisation only — labels come from the CSV) ─────────
ATLAS        = 'schaefer'   # 'schaefer' | 'destrieux' | 'custom'
N_ROIS       = 200
YEO_NETWORKS = 7
ATLAS_PATH   = None         # '/path/to/my_atlas.nii.gz'

# ── Thresholding ─────────────────────────────────────────────────────────────
# Proportion of edges to keep at the main analysis threshold.
# Typical range for resting-state FC: 0.05 – 0.25
THRESHOLD     = 0.10          # keep the strongest 10 % of edges
POSITIVE_ONLY = True          # discard negative edges before ranking

# Thresholds to test in the stability sweep
SWEEP_THRESHOLDS = (0.05, 0.10, 0.15, 0.20, 0.25)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(os.path.dirname(FC_MATRIX_PATH), 'graph_metrics')
os.makedirs(OUTPUT_DIR, exist_ok=True)

_sess_tag       = f'_{SESSION}' if SESSION else ''
OUTPUT_FILENAME = (f'{SUBJECT}{_sess_tag}_task-{TASK}'
                   f'_atlas-{ATLAS}{N_ROIS}'
                   f'_thr-{int(THRESHOLD*100):02d}pct_graph_metrics.csv')

print('Configuration ready.')
print(f'  FC matrix  : {FC_MATRIX_PATH}')
print(f'  Threshold  : {THRESHOLD} (keep top {int(THRESHOLD*100)}% of edges)')
print(f'  Pos only   : {POSITIVE_ONLY}')
print(f'  Output     : {OUTPUT_DIR}')

## 3. Explore the Helper Functions

Run the cells below to see what's available.  
The graph theory functions are listed under **Graph Theory**.

In [ ]:
fc_help()

In [ ]:
fc_help('threshold_proportional')

In [ ]:
# Try a few others:
# fc_help('matrix_to_graph')
# fc_help('compute_graph_metrics')
# fc_help('run_threshold_sweep')
# fc_help('plot_threshold_sweep')
# fc_help('plot_node_metric_by_network')
# fc_help('plot_degree_distribution')

---
## 4. Load the FC Matrix

The FC matrix saved by the FC tutorial is a labelled CSV: parcel names are both the row index and the column headers.  Read it back with `pd.read_csv(..., index_col=0)` and extract the labels and the numeric matrix.

**Returns:**
- `fc_df` — DataFrame shape (N_ROIS, N_ROIS)
- `roi_labels` — list of parcel label strings
- `fc_matrix` — NumPy array shape (N_ROIS, N_ROIS)

👉 *The matrix values are Fisher z-transformed Pearson correlations (unbounded, approximately normal). Positive values = correlated time series; negative = anti-correlated.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Load the FC matrix CSV.
# Hint: pd.read_csv(FC_MATRIX_PATH, index_col=0)
# Extract roi_labels from the column names (.tolist())
# Extract fc_matrix as a NumPy float array (.values)



# ─────────────────────────────────────────────────────────────────────────────
# After running: confirm shape and label format
# print(fc_matrix.shape)      # should be (N_ROIS, N_ROIS)
# print(roi_labels[:3])       # should look like '7Networks_LH_Vis_1'

---
## 5. Inspect the Full FC Matrix

Before thresholding, visualise the raw (unthresholded) FC matrix.  
You should see block structure corresponding to Yeo networks.

**Function:** `plot_fc_matrix(fc_matrix, labels, title, ...)`  

👉 *Call `fc_help('plot_fc_matrix')` for full parameter documentation.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Plot the unthresholded FC matrix using plot_fc_matrix().
# Pass: fc_matrix, labels=roi_labels, and a title.
# Hint: use vmin=-1, vmax=1 so the colour scale is centred.



# ─────────────────────────────────────────────────────────────────────────────
plt.show()
# After running: can you identify any network blocks along the diagonal?

---
## 6. Threshold the FC Matrix

**Function:** `threshold_proportional(fc_matrix, proportion, positive_only)`  
**Returns:** `adj` — thresholded adjacency matrix, same shape as input

We use a **proportional threshold**: keep the top X% of edges by weight, regardless of their absolute value. This ensures that subjects with different overall connectivity levels are compared at the same network density — a fairer comparison than an absolute weight threshold.

| Choice | Effect |
|--------|--------|
| Lower proportion | Sparser graph — only strongest hubs survive |
| Higher proportion | Denser graph — more connections, more noise |
| `positive_only=True` | Remove negative edges before ranking |

👉 *Call `fc_help('threshold_proportional')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Threshold the FC matrix using threshold_proportional().
# Pass: fc_matrix, proportion=THRESHOLD, positive_only=POSITIVE_ONLY
# Store result as: adj



# ─────────────────────────────────────────────────────────────────────────────
# Inspect the result
# n_edges  = int((adj > 0).sum() // 2)
# density  = n_edges / (adj.shape[0] * (adj.shape[0] - 1) / 2)
# print(f'Edges: {n_edges}  |  Density: {density:.4f}')

In [ ]:
# Visualise the thresholded adjacency matrix
fig = plot_fc_matrix(
    adj,
    labels  = roi_labels,
    title   = f'{SUBJECT} — Thresholded adjacency (top {int(THRESHOLD*100)}% edges)',
    cmap    = 'Reds',
    vmin    = 0,
)
plt.show()
# After running: compare the block structure to the unthresholded plot.
# Does the within-network connectivity dominate after thresholding?

---
## 7. Build a NetworkX Graph

**Function:** `matrix_to_graph(adj, labels)`  
**Returns:** `G` — a `networkx.Graph` with `label` and `network` node attributes

NetworkX represents the brain network as nodes (parcels) and edges (connections).  Once you have a graph object you can access the full NetworkX API for any metric not already in `fc_helpers`.

👉 *Call `fc_help('matrix_to_graph')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Build a NetworkX graph from the thresholded adjacency matrix.
# Pass: adj, labels=roi_labels
# Store result as: G



# ─────────────────────────────────────────────────────────────────────────────
# After running:
# print(f'Nodes : {G.number_of_nodes()}')
# print(f'Edges : {G.number_of_edges()}')
# print(f'Is connected: {nx.is_connected(G)}')
# Inspect a node's attributes:
# print(G.nodes[0])

---
## 8. Compute Node-Level and Graph-Level Metrics

**Function:** `compute_graph_metrics(adj, labels)`  
**Returns:** `node_df` (DataFrame, one row per parcel), `graph_dict` (summary dict)

**Node-level metrics** tell you about each individual parcel's role in the network:

| Metric | What it measures |
|--------|------------------|
| `degree` | Number of connections |
| `strength` | Sum of edge weights (weighted degree) |
| `clustering` | Tendency of a node's neighbours to also connect to each other |
| `betweenness` | Fraction of shortest paths that pass through this node |

**Graph-level metrics** summarise the whole network (one number per subject):
density, mean clustering, average path length, transitivity.

👉 *Call `fc_help('compute_graph_metrics')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Compute node and graph metrics using compute_graph_metrics().
# Pass: adj, labels=roi_labels
# Store results as: node_df, graph_dict



# ─────────────────────────────────────────────────────────────────────────────
# Inspect graph-level results
# print('\nGraph-level metrics:')
# for k, v in graph_dict.items():
#     print(f'  {k:<30s}: {v}')

# Inspect top-10 nodes by strength
# print('\nTop-10 nodes by strength:')
# print(node_df.nlargest(10, 'strength')[['label', 'network', 'strength', 'clustering']])

---
## 9. Degree Distribution

**Function:** `plot_degree_distribution(adj, bins)`  
**Returns:** `fig`, `degrees`

The degree distribution tells you whether the network has a few highly connected **hubs** (heavy tail) or whether connections are more evenly spread.  Scale-free networks (like some brain networks) show a power-law tail; random Erdős–Rényi graphs show a narrow Poisson distribution.

👉 *Call `fc_help('plot_degree_distribution')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Plot the degree distribution using plot_degree_distribution().
# Pass: adj
# Store results as: fig, degrees



# ─────────────────────────────────────────────────────────────────────────────
plt.show()
# After running: is the distribution skewed? Are there clear hub nodes?

---
## 10. Node Metrics by Yeo Network

**Function:** `plot_node_metric_by_network(node_df, metric, threshold)`  
**Returns:** `fig`

Aggregating node metrics by Yeo network lets you ask: *which functional networks are most strongly embedded in the graph?*  The default networks (DMN, frontoparietal) often show high strength and clustering in resting-state data.

Try plotting different metrics to see how they vary:
- `'strength'` — overall connectivity level per network
- `'clustering'` — local segregation within each network
- `'betweenness'` — which networks act as bridges between others

👉 *Call `fc_help('plot_node_metric_by_network')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Plot node strength by network using plot_node_metric_by_network().
# Pass: node_df, metric='strength'
# (Leave threshold=None — node_df already contains a single threshold.)



# ─────────────────────────────────────────────────────────────────────────────
plt.show()

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Now plot clustering coefficient by network.
# Does the network with the highest strength also have the highest clustering?



# ─────────────────────────────────────────────────────────────────────────────
plt.show()

---
## 11. Threshold Sweep — Stability Check

**Function:** `run_threshold_sweep(fc_matrix, labels, thresholds)`  
**Returns:** `node_sweep` (DataFrame), `graph_sweep` (DataFrame)

A single threshold is an arbitrary choice. Before trusting any result, run a **threshold sweep** to check that your conclusions hold across a range of sparsity levels.  If a result is only visible at one specific threshold, it is likely an artefact.

> **Rule of thumb:** prefer results that are stable across at least the 5–25% range.

**Function:** `plot_threshold_sweep(graph_sweep)`  
**Returns:** `fig`

👉 *Call `fc_help('run_threshold_sweep')` or `fc_help('plot_threshold_sweep')` for full docs.*

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Run the threshold sweep using run_threshold_sweep().
# Pass: fc_matrix, roi_labels, thresholds=SWEEP_THRESHOLDS
# Store results as: node_sweep, graph_sweep



# ─────────────────────────────────────────────────────────────────────────────
# After running: inspect graph_sweep
# print(graph_sweep[['threshold', 'density', 'mean_clustering',
#                     'avg_path_length', 'n_components']])

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# Plot the sweep results using plot_threshold_sweep().
# Pass: graph_sweep
# Store result as: fig



# ─────────────────────────────────────────────────────────────────────────────
plt.show()
# After running:
# - Does average path length increase as the graph gets sparser? (Expected: yes)
# - Is clustering relatively stable across thresholds?

---
## 12. Save Results

Save the node-level metrics and the threshold sweep summary as CSVs so they can be used in group-level analysis.

**Files to save:**
1. `node_df` — node metrics at the chosen `THRESHOLD`
2. `graph_sweep` — graph-level metrics across all sweep thresholds

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# 12a. Save node_df to OUTPUT_DIR / OUTPUT_FILENAME using node_df.to_csv()
# Hint: node_df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_FILENAME), index=False)



# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── YOUR CODE HERE ───────────────────────────────────────────────────────────
# 12b. Save graph_sweep with a descriptive filename.
# Hint: derive the filename by replacing '_graph_metrics' with '_graph_sweep'
#       in OUTPUT_FILENAME, or create a new name.



# ─────────────────────────────────────────────────────────────────────────────
# Verify
# check = pd.read_csv(os.path.join(OUTPUT_DIR, OUTPUT_FILENAME))
# print(f'Node metrics shape: {check.shape}')    # should be (N_ROIS, 6)

---
## 🔬 Checkpoint — Questions to Consider

Before moving to the answer key, think through these:

1. **Proportional vs absolute threshold** — Why do we prefer a proportional threshold for comparing subjects, even though it means each subject's graph has a slightly different set of edges?
2. **Positive-only edges** — Negative FC values are often retained in some analyses. What biological or methodological reasons might you have for keeping or discarding them?
3. **Clustering vs path length** — The *small-world* property requires high clustering AND short path lengths. Which Yeo networks contributed most to clustering in your graph?
4. **Disconnected components** — If `n_components > 1`, the graph is fragmented. How does `compute_graph_metrics` handle average path length in that case, and why is this important?
5. **Betweenness centrality** — A node with low strength but high betweenness acts as a **bridge** between communities. Which parcels have this property in your data?

---

# ✅ Answer Key

---
> **Scroll past here only after completing the exercise above.**
>
> The cells below show a complete, working solution for each step.
---

### AK-1. Load FC Matrix

In [ ]:
# ── ANSWER KEY: Section 4 — Load FC matrix ───────────────────────────────────
fc_df      = pd.read_csv(FC_MATRIX_PATH, index_col=0)
roi_labels = fc_df.columns.tolist()
fc_matrix  = fc_df.values.astype(float)

print(f'FC matrix shape : {fc_matrix.shape}')
print(f'Number of labels: {len(roi_labels)}')
print(f'First 3 labels  : {roi_labels[:3]}')
print(f'Value range     : [{fc_matrix.min():.3f}, {fc_matrix.max():.3f}]')

### AK-2. Inspect the Full FC Matrix

In [ ]:
# ── ANSWER KEY: Section 5 — Plot unthresholded FC matrix ─────────────────────
fig = plot_fc_matrix(
    fc_matrix,
    labels  = roi_labels,
    title   = f'{SUBJECT} — Unthresholded FC ({ATLAS.capitalize()}-{N_ROIS})',
    vmin    = -1,
    vmax    = 1,
    figsize = (10, 9),
)
plt.show()

### AK-3. Threshold the FC Matrix

In [ ]:
# ── ANSWER KEY: Section 6 — Threshold ────────────────────────────────────────
adj = threshold_proportional(
    fc_matrix,
    proportion    = THRESHOLD,
    positive_only = POSITIVE_ONLY,
)

n_edges = int((adj > 0).sum() // 2)
density = n_edges / (adj.shape[0] * (adj.shape[0] - 1) / 2)
print(f'Retained edges : {n_edges}')
print(f'Graph density  : {density:.4f}')

fig = plot_fc_matrix(
    adj,
    labels  = roi_labels,
    title   = f'{SUBJECT} — Thresholded adjacency (top {int(THRESHOLD*100)}%)',
    cmap    = 'Reds',
    vmin    = 0,
    figsize = (10, 9),
)
plt.show()

### AK-4. Build a NetworkX Graph

In [ ]:
# ── ANSWER KEY: Section 7 — Build NetworkX graph ─────────────────────────────
G = matrix_to_graph(adj, labels=roi_labels)

print(f'Nodes        : {G.number_of_nodes()}')
print(f'Edges        : {G.number_of_edges()}')
print(f'Is connected : {nx.is_connected(G)}')
print(f'Node 0 attrs : {G.nodes[0]}')

### AK-5. Compute Node and Graph Metrics

In [ ]:
# ── ANSWER KEY: Section 8 — Compute metrics ──────────────────────────────────
node_df, graph_dict = compute_graph_metrics(adj, labels=roi_labels)

print('\nGraph-level metrics:')
for k, v in graph_dict.items():
    print(f'  {k:<35s}: {v}')

print('\nTop-10 nodes by strength:')
print(node_df.nlargest(10, 'strength')[['label', 'network', 'strength',
                                         'clustering', 'betweenness']].to_string(index=False))

### AK-6. Degree Distribution

In [ ]:
# ── ANSWER KEY: Section 9 — Degree distribution ──────────────────────────────
fig, degrees = plot_degree_distribution(adj, bins=25)
plt.show()

print(f'Mean degree : {degrees.mean():.1f}')
print(f'Max degree  : {degrees.max()}')
print(f'Min degree  : {degrees.min()}')

# Top-5 hub nodes (highest degree)
hub_idx = np.argsort(degrees)[::-1][:5]
print('\nTop-5 hub nodes (by degree):')
for i in hub_idx:
    print(f'  {roi_labels[i]:<50s}  degree={degrees[i]}')

### AK-7. Node Metrics by Yeo Network

In [ ]:
# ── ANSWER KEY: Section 10 — Node metrics by network ─────────────────────────
fig = plot_node_metric_by_network(node_df, metric='strength')
plt.show()

fig = plot_node_metric_by_network(node_df, metric='clustering')
plt.show()

fig = plot_node_metric_by_network(node_df, metric='betweenness')
plt.show()

### AK-8. Threshold Sweep

In [ ]:
# ── ANSWER KEY: Section 11 — Threshold sweep ─────────────────────────────────
node_sweep, graph_sweep = run_threshold_sweep(
    fc_matrix,
    roi_labels,
    thresholds    = SWEEP_THRESHOLDS,
    positive_only = POSITIVE_ONLY,
)

print(graph_sweep[['threshold', 'density', 'mean_clustering',
                    'avg_path_length', 'n_components']].to_string(index=False))

In [ ]:
# ── ANSWER KEY: Section 11 continued — Plot sweep ───────────────────────────
fig = plot_threshold_sweep(graph_sweep)
sweep_fig_name = OUTPUT_FILENAME.replace('_graph_metrics', '_sweep')
sweep_fig_name = os.path.splitext(sweep_fig_name)[0] + '.png'  # figures must be image, not .csv
fig.savefig(os.path.join(OUTPUT_DIR, sweep_fig_name), dpi=150, bbox_inches='tight')
plt.show()

# Bonus: strength by network at each threshold from the sweep
for thr in SWEEP_THRESHOLDS:
    fig = plot_node_metric_by_network(
        node_sweep, metric='strength', threshold=thr,
        figsize=(9, 3.5),
    )
    plt.show()
    


### AK-9. Save Results

In [ ]:
# ── ANSWER KEY: Section 12 — Save ────────────────────────────────────────────

# Node metrics at the chosen threshold
node_out = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)
node_df.to_csv(node_out, index=False)
print(f'Saved node metrics : {node_out}')

# Graph-level sweep
sweep_out = os.path.join(
    OUTPUT_DIR,
    OUTPUT_FILENAME.replace('_graph_metrics', '_graph_sweep'),
)
graph_sweep.to_csv(sweep_out, index=False)
print(f'Saved sweep table  : {sweep_out}')

# Verify
check = pd.read_csv(node_out)
print(f'\nVerified node_df shape : {check.shape}')   # (N_ROIS, 6)
print(check[['label', 'network', 'degree', 'strength', 'clustering']].head())

---
## Summary

| Step | Helper function | Key output |
|------|-----------------|------------|
| Load FC matrix | `pd.read_csv()` | `fc_matrix` (N × N) + `roi_labels` |
| Visualise FC | `plot_fc_matrix()` | Heatmap with network blocks |
| Threshold | `threshold_proportional()` | `adj` (sparse adjacency) |
| Build graph | `matrix_to_graph()` | `G` (NetworkX graph) |
| Node metrics | `compute_graph_metrics()` | `node_df`, `graph_dict` |
| Degree distribution | `plot_degree_distribution()` | Histogram + hub nodes |
| Network summary | `plot_node_metric_by_network()` | Bar chart by Yeo network |
| Stability check | `run_threshold_sweep()` + `plot_threshold_sweep()` | `graph_sweep` |
| Save | `node_df.to_csv()` | `*_graph_metrics.csv` |

**Next steps:**
- Run the same pipeline on multiple subjects and compare `graph_dict` values across groups
- Use `node_sweep` to test whether network-level differences hold across thresholds
- Compute a random-graph null (same N and density) and compare clustering and path length → **small-world coefficient**
- Extend to gPPI connectivity matrices to ask whether task context changes network topology